# 03 - Xem từng bước của Dashboard

Notebook này gọi **đúng các module mà app dùng** (`src/data.py`, `factors.py`, `charts.py`, `llm.py`), nên mỗi bước ở đây ra gì thì app cũng ra như vậy. Dùng để quan sát, sửa và chụp hình đưa vào báo cáo 3.5.

> App thật chạy bằng `streamlit run dashboard/app.py` (Streamlit chỉ chạy file `.py`). Notebook chỉ để xem và thử.

**Chuẩn bị:** kernel `.venv-1`, đã `gcloud auth application-default login` (xem notebook 01).

In [14]:
%load_ext autoreload
%autoreload 2

2026-10-24 14:16:35.553 No runtime found, using MemoryCacheStorageManager
2026-10-24 14:16:35.554 No runtime found, using MemoryCacheStorageManager


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import sys, pathlib

# Tìm thư mục dashboard/ (chứa src/) để import được các module của app
here = pathlib.Path.cwd()
for base in [here, here.parent, here / "dashboard", here.parent / "dashboard"]:
    if (base / "src" / "config.py").exists():
        sys.path.insert(0, str(base))
        break
else:
    raise FileNotFoundError("Không tìm thấy dashboard/src. Hãy mở notebook từ bên trong repo.")
print("Dùng thư mục:", base)

Dùng thư mục: d:\olist-lakehouse-\ThanhTruc\dashboard_package\dashboard


In [16]:
import os
import pandas as pd
from IPython.display import display
from src import config as cfg, data, factors, charts, llm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", None)

## Bước 1. Tải dữ liệu (`src/data.py`)
JOIN bảng dự đoán với Wide Table (và `fct_orders` để có năm-tháng) theo `order_id`.

In [17]:
df = data.load_dashboard_data()
print("Kích thước:", df.shape)
print("Có thông tin năm-tháng:", charts.has_time(df))
display(df.head())

Kích thước: (99441, 24)
Có thông tin năm-tháng: True


,order_id,late_probability,predicted_late,risk_level,top_risk_factor_1,top_risk_factor_2,actual_late,customer_state,seller_state,product_category_name,purchase_month,purchase_weekday,item_count,order_value,freight_value,total_product_weight_g,max_product_weight_g,max_product_length_cm,max_product_height_cm,max_product_width_cm,seller_prep_days,distance_km,year_month,order_status
0,eb45ce36240dbf895bef98be0d1a85d3,0.4285,0,Medium Risk,distance_km,customer_state_AL,NaN,AL,SP,informatica_acessorios,10.0,5.0,1.0,32.50,20.80,200.0,200.0,36.0,9.0,20.0,5.028692,1745.093002,2016-10,invoiced
1,676157c2d1075c88ea0009f8d7d2d389,0.4311,0,Medium Risk,distance_km,customer_state_AL,0.0,AL,PR,automotivo,10.0,5.0,1.0,49.99,26.61,200.0,200.0,25.0,42.0,25.0,4.067106,2250.891518,2016-10,delivered
2,c4e980a1d822db426982878b3cfdda6e,0.6265,1,Medium Risk,distance_km,customer_state_BA,NaN,BA,PR,relogios_presentes,10.0,6.0,1.0,649.17,25.16,404.0,404.0,22.0,14.0,17.0,4.714387,1913.181269,2016-10,invoiced
3,ac00d3859c095d282f6bef69e0ed1b38,0.4204,0,Medium Risk,distance_km,customer_state_ES,0.0,ES,PR,cool_stuff,10.0,4.0,1.0,69.90,19.26,1333.0,1333.0,24.0,23.0,25.0,9.020162,1442.004297,2016-10,delivered
4,82e4f033f60598903866557d89c9c042,0.4613,0,Medium Risk,distance_km,customer_state_MA,0.0,MA,SP,utilidades_domesticas,10.0,6.0,1.0,19.90,24.84,200.0,200.0,16.0,4.0,11.0,4.025000,1928.789130,2016-10,delivered


In [18]:
print("Số đơn theo mức rủi ro:")
display(df["risk_level"].value_counts())
print("Số đơn chưa có nhãn thật:", int(df["actual_late"].isna().sum()))
print("Số đơn không có yếu tố rủi ro (factor_1 rỗng):", int(df["top_risk_factor_1"].isna().sum()))

Số đơn theo mức rủi ro:


risk_level
Low Risk       56868
Medium Risk    31401
High Risk      11172
Name: count, dtype: int64

Số đơn chưa có nhãn thật: 2965
Số đơn không có yếu tố rủi ro (factor_1 rỗng): 0


## Bước 2. Trung vị của các biến số
Dùng để nói 'cao hơn / thấp hơn mức thường' khi diễn giải yếu tố rủi ro.

In [19]:
medians = factors.compute_medians(df)
display(pd.Series(medians, name="trung vị").to_frame())

,trung vị
item_count,1.000000
order_value,86.900000
freight_value,17.170000
total_product_weight_g,750.000000
max_product_weight_g,700.000000
max_product_length_cm,25.000000
max_product_height_cm,13.000000
max_product_width_cm,20.000000
purchase_month,6.000000
purchase_weekday,3.000000


## Bước 3. Diễn giải yếu tố rủi ro (`src/factors.py`)
`top_risk_factor` lưu **tên đặc trưng kỹ thuật**. Với biến one-hot như `customer_state_SP`, điểm SHAP dương có thể nghĩa là khách **không** ở SP. Vì vậy phải so với giá trị thật của đơn. Xem 8 đơn nguy cơ cao nhất:

In [20]:
top8 = df[df["risk_level"] == cfg.RISK_HIGH].sort_values("late_probability", ascending=False).head(8)
for _, row in top8.iterrows():
    print(f"{row['order_id']}  (xác suất {factors.fmt_pct(row['late_probability'])}, khách {row['customer_state']}, người bán {row['seller_state']})")
    for i, f in enumerate(factors.describe_order(row, medians), 1):
        print(f"    {i}. [{row['top_risk_factor_' + str(i)]}]  ->  {f.text}")
    print()

4ef546f6f1165d23cd5134e9c5ab83d7  (xác suất 95,2%, khách RJ, người bán SP)
    1. [purchase_month]  ->  Đơn được đặt vào tháng 2
    2. [customer_state_RJ]  ->  Khách hàng ở bang RJ (Rio de Janeiro)

21d5a8c3c47b36c7e6a5d06cd1a89087  (xác suất 94,9%, khách SE, người bán SP)
    1. [purchase_month]  ->  Đơn được đặt vào tháng 3
    2. [max_product_length_cm]  ->  Chiều dài lớn nhất của sản phẩm: 105 cm (cao hơn mức trung vị 25 cm)

019d444bd00f1ccb4643de466f9a1525  (xác suất 94,8%, khách CE, người bán SP)
    1. [purchase_month]  ->  Đơn được đặt vào tháng 3
    2. [max_product_height_cm]  ->  Chiều cao lớn nhất của sản phẩm: 104 cm (cao hơn mức trung vị 13 cm)

e81fbdf6c9bbdc3b6a9cd243b617e576  (xác suất 94,7%, khách MA, người bán SP)
    1. [purchase_month]  ->  Đơn được đặt vào tháng 11
    2. [customer_state_MA]  ->  Khách hàng ở bang MA (Maranhão)

069514f19868af8a75ebe98f43d33900  (xác suất 94,7%, khách AL, người bán MA)
    1. [purchase_month]  ->  Đơn được đặt vào tháng 3
    2.

### Kiểm tra: có tên đặc trưng nào chưa được diễn giải không?

In [21]:
missing = sorted(set(factors.unknown_features(df["top_risk_factor_1"])) | set(factors.unknown_features(df["top_risk_factor_2"])))
print("Tên đặc trưng chưa biết cách diễn giải:", missing or "không có")

Tên đặc trưng chưa biết cách diễn giải: không có


## Bước 4. Chỉ số tổng quan (KPI)

In [22]:
kpis = charts.compute_kpis(df)
for k, v in kpis.items():
    print(f"{k:28s} {v}")

total_orders                 99441
labeled_orders               96476
actual_late_rate             0.06773705377503213
predicted_late_rate          0.22389155378566186
high_risk_orders             11172
high_risk_actual_late_rate   0.3309872198555288
avg_probability              0.35631811324580576


> Lưu ý khi đọc: **tỷ lệ mô hình dự báo trễ** thường cao hơn nhiều **tỷ lệ trễ thực tế** vì mô hình cố ý cảnh báo rộng để ít bỏ sót. Xem thêm *Đơn nguy cơ cao thật sự trễ* để biết độ tin cậy của cảnh báo.

## Bước 5. Biểu đồ (`src/charts.py`)

In [23]:
figs = {
    "Phân bố mức rủi ro": charts.fig_risk_distribution(df),
    "Tỷ lệ trễ theo bang": charts.fig_state_rates(df),
    "Đơn nguy cơ cao theo thời gian": charts.fig_high_risk_over_time(df),
    "Tỷ lệ trễ theo năm-tháng": charts.fig_late_rate_over_time(df),
    "Yếu tố rủi ro chính": charts.fig_top_factors(df[df["risk_level"] == cfg.RISK_HIGH], medians),
}
for name, fig in figs.items():
    print("==", name)
    if fig is None:
        print("   (không đủ dữ liệu để vẽ)")
    else:
        fig.show()

== Phân bố mức rủi ro


== Tỷ lệ trễ theo bang


== Đơn nguy cơ cao theo thời gian


== Tỷ lệ trễ theo năm-tháng


== Yếu tố rủi ro chính


## Bước 6. Prompt gửi cho Groq (`src/llm.py`)
Xem trước nội dung prompt của một đơn, **chưa gọi API**.

In [24]:
row = top8.iloc[0]
order_factors = factors.describe_order(row, medians)
print("--- SYSTEM ---")
print(llm.SYSTEM_PROMPT)
print("\n--- USER ---")
print(llm.build_user_prompt(row["order_id"], row["late_probability"], row["risk_level"], order_factors))

--- SYSTEM ---
Bạn là trợ lý hỗ trợ vận hành logistics cho sàn thương mại điện tử Olist (Brazil). Chỉ dựa vào thông tin được cung cấp trong yêu cầu; không bịa số liệu, không suy diễn nguyên nhân ngoài dữ liệu. Các yếu tố rủi ro cho biết điều gì làm tăng xác suất mô hình dự báo, không chứng minh quan hệ nhân quả. Trả lời bằng tiếng Việt, ngắn gọn, thực tế, dành cho nhân viên điều phối vận hành.

--- USER ---
Thông tin đơn hàng:
- Mã đơn: 4ef546f6f1165d23cd5134e9c5ab83d7
- Xác suất giao trễ do mô hình dự báo: 95,2%
- Mức rủi ro: Cao
- Các yếu tố làm tăng rủi ro (theo thứ tự quan trọng):
  1. Đơn được đặt vào tháng 2
  2. Khách hàng ở bang RJ (Rio de Janeiro)
Lưu ý: yếu tố 'tháng đặt hàng' chỉ cho biết tháng; không được suy luận đó là mùa cao điểm hay sự kiện nào.

Yêu cầu:
1. Giải thích 1-2 câu vì sao đơn này cần được chú ý.
2. Đề xuất 1-3 hành động cụ thể cho bộ phận vận hành (ví dụ: nhắc người bán chuẩn bị hàng sớm, chủ động báo khách thời gian giao dự kiến, theo dõi tuyến vận chuyển).

## Bước 7. Gọi Groq thật (cần API key)
Key được hỏi bằng ô nhập ẩn. Cùng một đơn gọi lần hai sẽ lấy từ bộ nhớ tạm, không tốn thêm lượt.

In [25]:
import getpass
os.environ["GROQ_API_KEY"] = os.environ.get("GROQ_API_KEY") or getpass.getpass("Dán GROQ_API_KEY (ẩn khi gõ): ").strip()

text, err = llm.generate_recommendation(
    row["order_id"], float(row["late_probability"]), row["risk_level"], order_factors,
)
print("LỖI:", err) if err else print(text)

2026-10-24 14:16:36.637 No runtime found, using MemoryCacheStorageManager
2026-10-24 14:16:36.639 No runtime found, using MemoryCacheStorageManager


Đơn 4ef546f6 có xác suất trễ 95,2% và rủi ro cao, vì được đặt vào tháng 2 và khách hàng ở bang RJ – hai yếu tố tăng khả năng chậm giao.

**Hành động đề xuất:**
1. Liên hệ ngay với người bán, yêu cầu chuẩn bị và xuất kho sớm nhất có thể.  
2. Gửi thông báo cho khách hàng tại RJ, thông báo thời gian giao dự kiến và khả năng trễ.  
3. Theo dõi lộ trình vận chuyển từ kho tới RJ, chuẩn bị phương án dự phòng (đổi nhà vận chuyển hoặc tăng ca nếu cần).


## Bước 8. Thử hỏi đáp về một đơn

In [26]:
context = llm.order_context(row, order_factors)
history = []
for q in ["Vì sao đơn này có nguy cơ trễ?", "Đơn này có được giảm giá không?"]:
    answer, err = llm.answer_question(context, history, q)
    print("Hỏi:", q); print("Đáp:", answer or err, "\n")
    history += [{"role": "user", "content": q}, {"role": "assistant", "content": answer or err}]

Hỏi: Vì sao đơn này có nguy cơ trễ?
Đáp: Đơn này được gán mức rủi ro **Cao** và xác suất giao trễ 95,2 % dựa trên các yếu tố sau (đều có trong dữ liệu đơn):

| Yếu tố | Tác động tăng rủi ro |
|--------|----------------------|
| **Tháng đặt hàng: 2** | Tháng 2 thường có nhu cầu giao hàng tăng (lễ hội, ngày nghỉ) và thời tiết có thể ảnh hưởng tới vận chuyển, làm tăng khả năng chậm trễ. |
| **Bang khách hàng: RJ (Rio de Janeiro)** | Vùng RJ được mô hình xác định là khu vực có tần suất trễ cao hơn so với một số bang khác, có thể do mật độ giao nhận lớn hoặc các yếu tố địa phương. |
| **Khoảng cách 596 km** | Khoảng cách trung bình‑cao giữa người bán (SP) và khách hàng (RJ) kéo dài thời gian vận chuyển và tăng khả năng gặp sự cố trên đường. |
| **Thời gian chuẩn bị 6 ngày** | Thời gian chuẩn bị dài (so với mức chuẩn thường 2‑3 ngày) làm giảm thời gian còn lại cho vận chuyển, khiến lịch trình gấp rút hơn. |
| **Giá trị đơn hàng thấp (10,99 BRL) & phí vận chuyển cao (15,10 BRL)** | Đơn hàng c

**Câu hỏi thứ hai cố ý hỏi điều không có trong dữ liệu** để kiểm tra trợ lý có từ chối trả lời bịa hay không. Nếu nó bịa, chỉnh `QA_SYSTEM_PROMPT` trong `src/llm.py`.

**Bước tiếp theo:** chạy app thật: `streamlit run dashboard/app.py`